# YOLO-World Zero-Shot Webcam
This notebook runs Ultralytics YOLO-World directly on your webcam. 

### Interactive Classes
It automatically creates a file named `classes.txt` in this folder. **You can open `classes.txt` in your editor, change the words, and save the file while the webcam is running.** The model will instantly update to detect your new classes!

In [7]:
import cv2
import time
import os
from IPython.display import clear_output
from ultralytics import YOLOWorld

In [8]:
# Initialize YOLO-World model (downloads yolov8s-worldv2.pt automatically ~20MB)
model = YOLOWorld("yolov8s-worldv2.pt")

# Create the classes.txt file if it doesn't exist
classes_file = "classes.txt"
if not os.path.exists(classes_file):
    with open(classes_file, "w") as f:
        f.write("person, cell phone, keyboard, bottle, cup")

print(f"Initial classes set! Edit '{classes_file}' to change them live.")

Initial classes set! Edit 'classes.txt' to change them live.


In [ ]:
print("Opening webcam...")
cap = cv2.VideoCapture(0)

target_fps = 10
frame_time = 1.0 / target_fps

last_mtime = 0
current_classes = []

if not cap.isOpened():
    print("Error: Could not open webcam.")
else:
    try:
        while True:
            start_time = time.time()
            
            # 1. LIVE UPDATE CLASSES FROM FILE
            try:
                current_mtime = os.path.getmtime(classes_file)
                if current_mtime != last_mtime:
                    last_mtime = current_mtime
                    with open(classes_file, "r") as f:
                        content = f.read()
                    new_classes = [c.strip() for c in content.split(',') if c.strip()]
                    if new_classes and new_classes != current_classes:
                        current_classes = new_classes
                        model.set_classes(current_classes)
            except Exception as e:
                pass # Ignore file read errors if saved exactly while reading

            # 2. READ WEBCAM
            success, frame = cap.read()
            if not success:
                print("Failed to read frame from webcam.")
                break

            # 3. RUN INFERENCE
            results = model.predict(frame, conf=0.1, verbose=False)

            # 4. PRINT DETECTIONS
            detected_indices = results[0].boxes.cls.cpu().tolist()
            
            clear_output(wait=True)
            print(f"Target Classes : {current_classes}")
            print("-" * 50)
            
            if detected_indices:
                detected_names = [results[0].names[int(idx)] for idx in detected_indices]
                counts = {}
                for name in detected_names:
                    counts[name] = counts.get(name, 0) + 1
                items_str = ", ".join([f"{k} ({v})" for k, v in counts.items()])
                print(f"I see         : {items_str}")
            else:
                print("I see         : nothing")
                
            print("-" * 50)
            print("Press 'q' on video to quit. Edit 'classes.txt' to change objects live.")

            # 5. RENDER FRAME
            annotated_frame = results[0].plot()
            cv2.imshow("Scamtir - YOLO-World Zero-Shot", annotated_frame)

            # 6. BREAK / FPS LIMIT
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
                
            elapsed = time.time() - start_time
            sleep_time = frame_time - elapsed
            if sleep_time > 0:
                time.sleep(sleep_time)
                
    finally:
        # Clean up and force window destruction in Jupyter
        cap.release()
        cv2.destroyAllWindows()
        cv2.waitKey(1)
        clear_output(wait=True)
        print("Webcam closed and window auto-destroyed.")

Webcam closed and window auto-destroyed.


KeyboardInterrupt: 